In [ ]:
%matplotlib inline
import scanpy as sc
import squidpy as sq
import DeepTalk_ST as dt
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)  
%matplotlib inline
%reload_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd

try:
    adata_st = sq.datasets.visium_fluo_adata_crop()  
    adata_st = adata_st[adata_st.obs.cluster.isin([f"Cortex_{i}" for i in np.arange(1, 5)])].copy()  
    img = sq.datasets.visium_fluo_image_crop()  
    adata_sc = sq.datasets.sc_mouse_cortex()  

except Exception as e:
    print(f"Error occurred: {e}")

adata_sc = adata_sc[
    adata_sc.obs['pct_counts_mt'] < 20]  
adata_sc = adata_sc[adata_sc.obs['n_genes_by_counts'] < 16000]  
adata_sc = adata_sc[
    adata_sc.obs['total_counts'] < 55000]  

sc.pp.filter_cells(adata_sc, min_genes=200)  
sc.pp.filter_genes(adata_sc, min_cells=3)  
sc.pp.normalize_total(adata_sc)  
sc.tl.rank_genes_groups(adata_sc, groupby="cell_subclass", use_raw=False)

markers_df = pd.DataFrame(adata_sc.uns["rank_genes_groups"]["names"]).iloc[0:250,:]  
markers = list(np.unique(markers_df.melt().value.values))  
len(markers)  
dt.pp_adatas(adata_sc, adata_st, genes=markers)